## notas

- criar funcao que tem como parametro o texto e a % de palavras

- melhorar distribuicao dos nodes

- aplicar à app

carregar libraries

In [25]:
import networkx as nx
import plotly.graph_objects as go
import numpy as np
import matplotlib.pyplot as plt
import base64
from io import BytesIO
import json

def rgb_string_to_hex(rgb_string):
    rgb_values = rgb_string.strip('rgb()').split(',')
    rgb = tuple(int(value.strip()) for value in rgb_values)
    return '#{:02X}{:02X}{:02X}'.format(rgb[0], rgb[1], rgb[2])

dependencias para a funcao: dados, numero de palavras a apresentar, query

In [26]:
with open("cache/7ab9dd2c21.json", "r") as f:
    data = json.load(f)

numero_de_palavras = 50

query = "QUERY"

tratar dados e definir intervalos do sentimento

In [27]:
# Dependecie for: how many words and sentiment intervals
counts = set()
sentiments = []
for word in data:
    counts.add(data[word]["count"])
    sentiments.append(data[word]["sentiment"])
sorted_counts = sorted(counts, reverse=True)

# Filter the data
data = {k: v for k, v in data.items() if v["count"] > sorted_counts[numero_de_palavras]}

# Dependecie for: sentiment intervals
sentiments2 = []
for word in data:
    sentiments2.append(data[word]["sentiment"])

# Set sentiment intervals
quantile10 = np.quantile(sentiments, 0.1)*0.4 + np.quantile(sentiments2, 0.1)*0.6
quantile40 = np.quantile(sentiments, 0.3)*0.3 + np.quantile(sentiments2, 0.3)*0.7
quantile60 = np.quantile(sentiments, 0.7)*0.3 + np.quantile(sentiments2, 0.7)*0.7
quantile90 = np.quantile(sentiments, 0.9)*0.4 + np.quantile(sentiments2, 0.9)*0.6

criar a base do grafo

In [28]:
# Create the graph
G = nx.Graph()

# Add nodes to the graph with attributes
for word, attributes in data.items():
    G.add_node(word, **attributes)

# Node positions
pos = nx.spring_layout(G, seed=124348)
pos = {node: np.random.rand(2) for node in G.nodes}

# Lists for node info
node_x, node_y = [], []
node_text, node_form = [], []
node_size, node_color= [], []
node_hovertext, custom_data = [], []

informacoes para cada node

In [29]:
# Populate node information
for node in G.nodes:
    
    # Node position
    x, y = pos[node]
    node_x.append(x)
    node_y.append(y)

    # Node text (name)
    if " " in node:
        splitted_text = node.split(" ")
        mid_text = len(splitted_text)//2
        node_text.append(' '.join(splitted_text[:mid_text]) + '<br>' + ' '.join(splitted_text[mid_text:]))
    else:
        node_text.append(node)

    # Node form
    node_form.append("circle")

    # Node size
    node_size.append(np.log(G.nodes[node]["count"])**1.3*5)
    
    # Node color
    sentiment = G.nodes[node]["sentiment"]
    if sentiment <= quantile10:
        sentiment_class = "muito negativo"
        color = "rgb(204, 0, 0)"
        node_color.append(color)
    elif sentiment <= quantile40:
        sentiment_class = "negativo"
        color = "rgb(239, 83, 80)"
        node_color.append(color)
    elif sentiment < quantile60:
        sentiment_class = "neutro"
        color = "rgb(204, 204, 204)"
        node_color.append(color)
    elif sentiment < quantile90:
        sentiment_class = "positivo"
        color = "rgb(102, 187, 106)"
        node_color.append(color)
    else:
        sentiment_class = "muito positivo"
        color = "rgb(0, 200, 81)"
        node_color.append(color)

    # Dependecie for: Node hovertext (last said) and Custom data (websites and plot)
    websites = sorted(G.nodes[node]["news"], reverse=True)
    first_website = websites[-1].split("/")
    first_website_date = first_website[5][:4] + "/" + first_website[5][4:6]
    last_website = websites[0].split("/")
    last_website_date = last_website[5][:4] + "/" + last_website[5][4:6]

    # Node hovertext
    last_time_said = max(int(key) for key, value in G.nodes[node]["date"].items() if value is not None)
    node_hovertext.append(
        f"""Palavra: {node}
        <br>Menções: {int(G.nodes[node]['count'])}
        <br>Último registo: {last_website_date}"""
    )

    # Dependecie for: Custom data (websites)
    websites_data = ""
    for website in websites:
        website_ = website.split("/")
        websites_data += f"<p><a href='{website}' target='_blank'>{website_[5][:4]+'/'+website_[5][4:6]+' - '+ '/'.join(website_[6:])}</a></p>"

    # Dependecie for: Custom data (plot)
    times_said_by_year = {str(k): 0 for k in range(int(first_website_date[:4]),
                                              int(last_website_date[:4])+1)}
    for key in G.nodes[node]["date"].keys():
        times_said_by_year[key[:4]] += int(G.nodes[node]["date"][key])
    plt.figure(figsize=(6, 4))
    plt.bar(times_said_by_year.keys(), times_said_by_year.values(), color=rgb_string_to_hex(color))
    plt.xlabel('Ano')
    plt.ylabel('Número de Menções')
    plt.grid(axis='y', alpha=0.2)
    plt.tight_layout()
    buffer = BytesIO()
    plt.savefig(buffer, format='png', transparent=True)
    plt.close() 
    buffer.seek(0)
    img_str = base64.b64encode(buffer.read()).decode('utf-8')

    # Dependecie for: Custom data (source)
    source = G.nodes[node]["source"]
    source_data = ""
    for key in source.keys():
        if source[key] is not None:
            source_data += f"<li>{key}: {int(source[key])}</li>"
        
    # Node custom data
    custom_data.append(f"""
        <h2 style="text-align: center;">Associação entre<br>{query} e {node}</h2>
        <p>Menções: {int(G.nodes[node]['count'])}</p>
        <p>Sentimento: {sentiment_class}</p>
        <p>Fontes:</p>
        <ul>
        {source_data}
        </ul>
        <div class="url-navigation">
            <button onclick="navigateUrl(-1)">&lt;</button>
            <span id="current-url"></span>
            <button onclick="navigateUrl(1)">&gt;</button>
        </div>
        <div id="website-urls">
            {websites_data}
        </div>
        <img src="data:image/png;base64,{img_str}" alt="Bar Plot" style="width:100%; height:auto;">
        """)

gerar grafo e algumas definicoes

In [30]:
# Create the Plotly figure
fig = go.Figure()

# Draw nodes
fig.add_trace(
    go.Scatter(
        x=node_x,
        y=node_y,
        mode="markers+text",
        text=node_text,
        hovertext=node_hovertext,
        marker=dict(
            color=node_color,
            symbol=node_form,
            size=node_size,
            line=dict(color="black", width=1)
        ),
        hoverinfo="text",
        customdata=custom_data,
        hoverlabel=dict(
            font=dict(color="rgb(48, 62, 92)"),
            bordercolor="rgb(48, 62, 92)"
        )
    )
)

# Update the layout
fig.update_layout(
    showlegend=False,
    xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    margin=dict(l=0, r=0, t=0, b=0)  # Remove all margins
)

""

''

algumas alteracoes no grafo, com css e js

In [31]:
# Generate the base HTML with graph
html_code = fig.to_html(include_plotlyjs='inline', full_html=True)

# Some CSS and JavaScript code to create the info panel
additional_html = """
<style>
    body, html {
        height: 100%;
        margin: 0;
        font-family: Arial, sans-serif;
        overflow: hidden;
    }
    #graph {
        position: absolute;
        top: 0;
        left: 0;
        right: 0;
        bottom: 0;
        z-index: 1; /* Ensure the graph is behind the panel */
    }
    #info-panel {
        position: absolute;
        overflow-y: auto;
        right: 0; /* Default to right */
        top: 0;
        width: 300px; /* Fixed width of the panel */
        height: 100vh; /* Full height of container */
        background-color: #f4f4f9;
        box-shadow: -2px 0 5px rgba(0,0,0,0.1);
        transform: scale(0); /* Hide initially */
        transition: transform 0.3s ease; /* Smooth slide in */
        z-index: 2; /* Panel above the graph */
        display: flex;
        flex-direction: column; /* Arrange children vertically */
    }
    #info-panel.open {
        transform: scale(1); /* Slide in the panel */
    }
    .close-button {
        background-color: #ff4c4c;
        color: white;
        border: none;
        padding: 10px;
        cursor: pointer;
        float: right;
        margin-bottom: -15px;
    }

    .close-button:hover {
        background-color: #e04343;
    }

    #website-urls {
        flex: 1; /* Take remaining vertical space */
        max-height: auto; /* Adjust height as needed */
        overflow-x: auto;
        padding: 4px; /* Padding for inner content */
        background-color: #ffffff; /* Background color */
        border: 1px solid #ddd; /* Border for the scrollable area */
        box-shadow: inset 0 0 5px rgba(0,0,0,0.1); /* Inner shadow */
        margin-top: 7px; /* Spacing from the title */
    }
    
    #website-urls p {
        white-space: nowrap; /* Prevents line breaks within the item */
    }

    p {
        margin: 5px 5px;
    }

    ul {
        margin-top: 0; /* Set the top margin of the unordered list to 0 */
    }

    li {
        margin-bottom: 5px;
    }

    .url-navigation button {
        border: none;
        border-radius: 2px;
        cursor: pointer;
        background-color: #6c757d;
        color: white;
        transition: background-color 0.3s;
    }

    .url-navigation button:hover {
        background-color: #5a6268;
    }
</style>

<div id="info-panel">
    <button class="close-button" onclick="closePanel()">Fechar</button>
    <p id="node-info">Escolha um nó para ver detalhes.</p>
</div>

<script>
    let currentUrlIndex = 0; // Global variable to track the currently displayed URL

    // Function to close the info panel
    function closePanel() {
        var panel = document.getElementById('info-panel');
        if (panel.classList.contains('open')) {
            panel.style.transform = 'scale(0)';
            setTimeout(() => {
                panel.classList.remove('open'); 
            }, 300); 
        }
    }

    function navigateUrl(direction) {
        const urls = document.querySelectorAll("#website-urls p");
        if (urls.length === 0) return;

        // Hide the currently visible URL
        urls[currentUrlIndex].style.display = "none";

        // Update the index based on the direction
        currentUrlIndex += direction;

        // Wrap around if out of bounds
        if (currentUrlIndex < 0) {
            currentUrlIndex = urls.length - 1; // Go to the last URL if moving left
        } else if (currentUrlIndex >= urls.length) {
            currentUrlIndex = 0; // Go back to the first URL if moving right
        }

        // Show the new URL
        urls[currentUrlIndex].style.display = "block";

        // Update the display span with the current index
        document.getElementById("current-url").textContent = `Notícia ${currentUrlIndex + 1}/${urls.length}`;
    }

    // Function to initialize the URL display for a new node
    function initializeUrls() {
        const urls = document.querySelectorAll("#website-urls p");
        urls.forEach((url) => {
            url.style.display = "none"; // Hide all URLs initially
        });

        // Reset index and show the first URL if available
        currentUrlIndex = 0; 
        if (urls.length > 0) {
            urls[currentUrlIndex].style.display = "block"; // Show the first URL
        }
        
        // Update the display span with the initial index
        document.getElementById("current-url").textContent = `Notícia ${currentUrlIndex + 1}/${urls.length}`;
    }

    document.addEventListener('DOMContentLoaded', function() {
        var plotDiv = document.querySelector('.plotly-graph-div');
        
        if (plotDiv) {
            plotDiv.on('plotly_click', function(data) {
                if (data.points.length > 0) {
                    var point = data.points[0];
                    var customData = point.customdata; 

                    document.getElementById('node-info').innerHTML = customData; // Use custom data for the panel

                    // Reset the URL index when a new node is clicked
                    currentUrlIndex = 0; // Reset to first URL
                    try {
                        initializeUrls(); // Reinitialize the URLs
                    } catch (error) {
                        // Code to handle the error
                        console.error("An error occurred:", error);
                    }

                    // Determine mouse click position for panel placement
                    var mouseX = data.event.clientX; 
                    var panel = document.getElementById('info-panel');

                    // Adjust panel position based on mouse click
                    if (mouseX > window.innerWidth / 2) {
                        panel.style.left = '0'; 
                        panel.style.right = 'auto'; 
                    } else {
                        panel.style.right = '0'; 
                        panel.style.left = 'auto'; 
                    }

                    panel.classList.add('open'); 
                    panel.style.transform = 'translateX(0)'; // Animate to the open position
                }
            });
        } else {
            console.error("Graph div not found.");
        }
    });
</script>
"""

# Combine the HTML and CSS/JS code
final_html = html_code.replace("</body>", additional_html + "</body>")

guardar o grafo

In [32]:
with open("graph_galptest.html", 'w') as f:
    f.write(final_html)

print(f"Graph file saved as graph_galptest.html")

Graph file saved as graph_galptest.html
